In [1]:
%load_ext autoreload
%autoreload 2

import polars as pl

from hea import lm, glm, data
from hea.family import binomial
from hea.compare import AIC, anova, drop1, step

## Examples from Extending the Linear Model with R, 2ed (Farawary, 2016)

### Ch1

In [2]:
gavote = data("gavote")
gavote.summary()

    rowname        equip      econ         perAA         rural       atlanta
Length:      159  LEVER:74  middle:69  Min.   :0.0000  rural:117  Atlanta   : 15
Class :character  OS-CC:44  poor  :72  1st Qu.:0.1115  urban: 42  notAtlanta:144
Mode  :character  OS-PC:22  rich  :18  Median :0.2330
                  PAPER: 2             Mean   :0.2430
                  PUNCH:17             3rd Qu.:0.3480
                                       Max.   :0.7650

     gore            bush           other           votes          ballots
Min.   :   249  Min.   :   271  Min.   :   5.0  Min.   :   832  Min.   :   881
1st Qu.:  1386  1st Qu.:  1804  1st Qu.:  30.0  1st Qu.:  3506  1st Qu.:  3694
Median :  2326  Median :  3597  Median :  86.0  Median :  6299  Median :  6712
Mean   :  7020  Mean   :  8929  Mean   : 381.7  Mean   : 16331  Mean   : 16927
3rd Qu.:  4430  3rd Qu.:  7468  3rd Qu.: 210.0  3rd Qu.: 11846  3rd Qu.: 12251
Max.   :154509  Max.   :140494  Max.   :7920.0  Max.   :263211  Max.   :28

In [3]:
gavote = (
    gavote
        .mutate(usage = pl.col("rural"))
        .select(pl.exclude("rural"))
        .mutate(
        undercount = (pl.col("ballots") - pl.col("votes")) / pl.col("ballots"),
        pergore = pl.col("gore") / pl.col("votes"),
    )
)
gavote.select("pergore", "undercount").describe()

statistic,pergore,undercount
str,f64,f64
"""count""",159.0,159.0
"""null_count""",0.0,0.0
"""mean""",0.408322,0.043792
"""std""",0.106833,0.024965
"""min""",0.187219,0.0
"""25%""",0.332444,0.027951
"""50%""",0.387447,0.039833
"""75%""",0.481868,0.056668
"""max""",0.782242,0.188121


In [4]:
m0_ch1 = lm('undercount ~ pergore + perAA', data=gavote)
m0_ch1.summary()

Formula: undercount ~ pergore + perAA

Residuals:
      Min        1Q    Median       3Q      Max
-0.046013 -0.014995 -0.003539 0.011784 0.142436

Coefficients:
             Estimate  Std. Error   CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)   0.03238     0.01276   0.007169   0.057583   2.5370    0.0122  *
pergore       0.01098     0.04692  -0.081706   0.103663   0.2340    0.8153
perAA         0.02853     0.03074  -0.032183   0.089249   0.9283    0.3547
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 159, p = 3, Residual SE = 0.024 on 156 DF
R-Squared = 0.0531, adjusted R-Squared = 0.0409
F-statistics = 4.3731 on 2 and 156 DF, p-value: 0.01

Log Likelihood = 365.9788, AIC = -723.9576, BIC = -711.6820


In [5]:
gavote = gavote.mutate(
    cpergore = pl.col("pergore") - pl.col("pergore").mean(),
    cperAA = pl.col("perAA") - pl.col("perAA").mean(),
)
m1_ch1 = lm('undercount ~ cperAA + cpergore*usage + equip', data=gavote)
m1_ch1.summary()

Formula: undercount ~ cperAA + cpergore*usage + equip

Residuals:
      Min        1Q    Median       3Q      Max
-0.059530 -0.012904 -0.002180 0.009013 0.127496

Coefficients:
                      Estimate  Std. Error    CI[2.5%]   CI[97.5]%  t value  Pr(>|t|)
(Intercept)           0.043297    0.002839   0.0376884   0.0489062  15.2529   < 2e-16  ***
cperAA                0.028264    0.031092  -0.0331711   0.0896992   0.9090   0.36479
cpergore              0.008237    0.051156  -0.0928429   0.1093166   0.1610   0.87230
usageurban           -0.018637    0.004648  -0.0278209  -0.0094523  -4.0095  9.56e-05  ***
equipOS-CC            0.006482    0.004680  -0.0027646   0.0157296   1.3852   0.16806
equipOS-PC            0.015640    0.005827   0.0041252   0.0271540   2.6838   0.00810  **
equipPAPER           -0.009092    0.016926  -0.0425368   0.0243528  -0.5372   0.59196
equipPUNCH            0.014150    0.006783   0.0007477   0.0275515   2.0861   0.03866  *
cpergore:usageurban  -0.008799  

In [6]:
anova(m0_ch1, m1_ch1)

Analysis of Variance Table

m0_ch1: undercount ~ pergore + perAA
m1_ch1: undercount ~ cperAA + cpergore*usage + equip

        Res.Df    RSS  Df  Sum of Sq      F    Pr(>F)
m0_ch1     156  0.093
m1_ch1     150  0.082   6      0.011  3.508  0.002823  **
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [7]:
drop1(m1_ch1, test="F")

Single term deletions

Model:
undercount ~ cperAA + cpergore*usage + equip

                Df  Sum of Sq     RSS         AIC  F value   Pr(>F)
<none>                         0.0818  -1186.0565
cperAA           1     0.0005  0.0822  -1187.1829   0.8264  0.36480
equip            4     0.0054  0.0872  -1183.8093   2.4964  0.04521  *
cpergore:usage   1     0.0000  0.0818  -1188.0017   0.0517  0.82050
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [8]:
m2_ch1 = lm('undercount ~ poly(cperAA, 4) + cpergore * usage + equip', data=gavote)
m2_ch1.summary()

Formula: undercount ~ poly(cperAA, 4) + cpergore * usage + equip

Residuals:
      Min        1Q    Median       3Q      Max
-0.058563 -0.012963 -0.001987 0.009230 0.127984

Coefficients:
                      Estimate  Std. Error    CI[2.5%]   CI[97.5]%  t value  Pr(>|t|)
(Intercept)           0.043460    0.002875   0.0377775   0.0491419  15.1150   < 2e-16  ***
poly(cperAA, 4)1      0.052258    0.069391  -0.0848750   0.1893914   0.7531   0.45260
poly(cperAA, 4)2     -0.002988    0.026135  -0.0546366   0.0486610  -0.1143   0.90914
poly(cperAA, 4)3     -0.005363    0.024267  -0.0533199   0.0425931  -0.2210   0.82538
poly(cperAA, 4)4     -0.016513    0.024199  -0.0643367   0.0313099  -0.6824   0.49606
cpergore              0.013153    0.056930  -0.0993540   0.1256595   0.2310   0.81761
usageurban           -0.019129    0.004741  -0.0284989  -0.0097598  -4.0348  8.76e-05  ***
equipOS-CC            0.006440    0.004720  -0.0028883   0.0157675   1.3643   0.17455
equipOS-PC            0.0155

In [9]:
m3_ch1 = lm('undercount ~ (equip + econ + usage + atlanta)^2 + (equip + econ + usage + atlanta) * (perAA + pergore)', data=gavote)
m3_ch1.summary()


Formula: undercount ~ (equip + econ + usage + atlanta)^2 + (equip + econ + usage + atlanta) * (perAA + pergore)

Residuals:
      Min        1Q    Median       3Q      Max
-0.055158 -0.008735 4.163e-17 0.008582 0.071595

Coefficients: (7 not defined because of singularities)
                                Estimate  Std. Error   CI[2.5%]  CI[97.5]%   t value  Pr(>|t|)
(Intercept)                    0.1133313   0.1273458  -0.138848   0.365511   0.88995    0.3753
equipOS-CC                    -0.0130587   0.0550868  -0.122146   0.096028  -0.23706    0.8130
equipOS-PC                    -0.1239885   0.0609227  -0.244632  -0.003345  -2.03518    0.0441  *
equipPAPER                    -0.0564436   0.0377654  -0.131229   0.018342  -1.49458    0.1377
equipPUNCH                    -0.0629444   0.0536073  -0.169102   0.043213  -1.17418    0.2427
econpoor                       0.0219635   0.0345227  -0.046401   0.090328   0.63620    0.5259
econrich                      -0.0806679   0.1421211  -0

In [10]:
m3_ch1_small = step(m3_ch1, trace=False)

In [11]:
drop1(m3_ch1_small, test="F")

Single term deletions

Model:
undercount ~ 1 + equip + econ + usage + perAA + equip:econ + equip:perAA + usage:perAA

             Df  Sum of Sq     RSS         AIC  F value    Pr(>F)
<none>                      0.0536  -1231.1419
equip:econ    6     0.0075  0.0612  -1222.2683   3.2500  0.005084  **
equip:perAA   4     0.0068  0.0605  -1220.0446   4.4348  0.002101  **
usage:perAA   1     0.0010  0.0546  -1230.1421   2.6474  0.106000
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [12]:
m4_ch1 = lm('undercount ~ equip + econ + perAA + equip:econ + equip:perAA', data=gavote)
m4_ch1.summary()

Formula: undercount ~ equip + econ + perAA + equip:econ + equip:perAA

Residuals:
      Min        1Q     Median       3Q      Max
-0.057011 -0.010632 -6.898e-05 0.009198 0.082545

Coefficients: (2 not defined because of singularities)
                      Estimate  Std. Error   CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)           0.041871    0.005028   0.031932   0.051810   8.3282  6.50e-14  ***
equipOS-CC           -0.011327    0.007373  -0.025902   0.003248  -1.5363  0.126700
equipOS-PC            0.008575    0.011178  -0.013523   0.030673   0.7671  0.444288
equipPAPER           -0.058427    0.037014  -0.131602   0.014747  -1.5785  0.116687
equipPUNCH           -0.015751    0.018745  -0.052810   0.021307  -0.8403  0.402175
econpoor              0.020266    0.005529   0.009335   0.031196   3.6654  0.000349  ***
econrich             -0.016966    0.012392  -0.041464   0.007532  -1.3692  0.173128
perAA                -0.042040    0.016594  -0.074845  -0.009236  -2.5335  0.01238

### Ch2

In [13]:
wcgs = data("wcgs", package="faraway")
wcgs.select("chd", "height", "cigs").summary()

  chd        height          cigs
no :2897  Min.   :60.00  Min.   : 0.00
yes: 257  1st Qu.:68.00  1st Qu.: 0.00
          Median :70.00  Median : 0.00
          Mean   :69.78  Mean   :11.60
          3rd Qu.:72.00  3rd Qu.:20.00
          Max.   :78.00  Max.   :99.00

In [14]:
m1_ch2 = glm('chd ~ height + cigs', family=binomial(), data=wcgs)
m1_ch2.summary()

Call:
glm(formula = chd ~ height + cigs, family = binomial(link=logit))

Coefficients:
              Estimate  Std. Error  CI[2.5%]  CI[97.5]%  z value  Pr(>|z|)
(Intercept)  -4.501614    1.841863  -8.11160   -0.89163  -2.4441    0.0145  *
height        0.025208    0.026327  -0.02639    0.07681   0.9575    0.3383
cigs          0.023127    0.004040   0.01521    0.03105   5.7243  1.04e-08  ***
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1; fixed)

    Null deviance: 1781.2437  on 3153 degrees of freedom
Residual deviance: 1749.0492  on 3151 degrees of freedom
AIC: 1755.0492    BIC: 1773.2185    logLik: -874.5246

Number of Fisher Scoring iterations: 5


In [15]:
m2_ch2 = glm('chd ~ cigs', family=binomial(), data=wcgs)
anova(m2_ch2, m1_ch2, test="Chisq")

Analysis of Deviance Table

m2_ch2: chd ~ cigs
m1_ch2: chd ~ height + cigs

        Resid. Df  Resid. Dev  Df  Deviance  Pr(>Chi)
m2_ch2       3152   1749.9695
m1_ch2       3151   1749.0492   1    0.9202    0.3374
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [16]:
drop1(m1_ch2, test="chisq")

Single term deletions

Model:
chd ~ height + cigs

        Df   Deviance        AIC      LRT  Pr(>Chi)
<none>      1749.0492  1755.0492
height   1  1749.9695  1753.9695   0.9202    0.3374
cigs     1  1780.1187  1784.1187  31.0695  2.49e-08  ***
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [17]:
wcgs = wcgs.mutate(
    bmi=703* pl.col("weight")/(pl.col("height")**2)
)
m3_ch2 = glm('chd ~ age + height + weight + bmi + sdp + dbp + chol + dibep + cigs + arcus',
             family=binomial(), data=wcgs)
m3r_ch2 = step(m3_ch2, trace=0)
m3r_ch2.summary()

Call:
glm(formula = chd ~ 1 + age + height + bmi + sdp + chol + dibep + cigs + arcus, family = binomial(link=logit))

Coefficients:
                Estimate  Std. Error    CI[2.5%]   CI[97.5]%  z value  Pr(>|z|)
(Intercept)   -15.957599    2.286076  -20.438226  -11.476972   -6.980  2.94e-12  ***
age             0.061590    0.012397    0.037293    0.085888    4.968  6.76e-07  ***
height          0.050161    0.027824   -0.004372    0.104694    1.803    0.0714  .
bmi             0.060385    0.026599    0.008252    0.112517    2.270    0.0232  *
sdp             0.017728    0.004155    0.009585    0.025871    4.267  1.98e-05  ***
chol            0.010709    0.001529    0.007713    0.013705    7.006  2.45e-12  ***
dibepB          0.657616    0.145898    0.371660    0.943571    4.507  6.56e-06  ***
cigs            0.021041    0.004262    0.012686    0.029395    4.936  7.96e-07  ***
arcuspresent    0.210998    0.143718   -0.070683    0.492680    1.468    0.1421
---
Signif. codes: 0 '***' 0.001

In [18]:
drop1(glm('chd ~ dbp', family=binomial(), data=wcgs), test="Chisq")

Single term deletions

Model:
chd ~ dbp

        Df   Deviance        AIC      LRT   Pr(>Chi)
<none>      1751.6955  1755.6955
dbp      1  1781.2437  1783.2437  29.5482  5.454e-08  ***
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [19]:
iris = data("iris")
irisr = (
    iris
    .filter(pl.col("Species") != "virginica")
    .select("Sepal.Width", "Sepal.Length", "Species")
)

In [20]:
m4_ch2 = glm('Species ~ Sepal.Width + Sepal.Length', family=binomial(), data=irisr)
m4_ch2.summary()

Call:
glm(formula = Species ~ Sepal.Width + Sepal.Length, family = binomial(link=logit))

Coefficients:
              Estimate  Std. Error   CI[2.5%]  CI[97.5]%    z value  Pr(>|z|)
(Intercept)     -360.6    195971.7  -384458.1   383736.9  -0.001840     0.999
Sepal.Width     -110.1     55361.0  -108615.7   108395.5  -0.001989     0.998
Sepal.Length     131.8     64576.5  -126435.8   126699.4   0.002041     0.998
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1; fixed)

    Null deviance: 138.6294  on 99 degrees of freedom
Residual deviance: 0.0000  on 97 degrees of freedom
AIC: 6.0000    BIC: 13.8155    logLik: -0.0000

Number of Fisher Scoring iterations: 25  (did NOT converge!)
